# Infaction World Generator v0.5.0 — Zone Framework
Same source as the MVP. Seeded results include six Factions and permanent Basic/Composite Zones across Inside City, Outside City and Secret Zone. This notebook intentionally exposes secret data for design review.


In [ ]:
import urllib.request, pathlib, subprocess, json, shutil
if not shutil.which('node'):
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','nodejs'],check=True)
url = 'https://raw.githubusercontent.com/playrenathea/rantau/main/world-generator.js'
pathlib.Path('world-generator.js').write_bytes(urllib.request.urlopen(url).read())
def generate(seed=42):
    return json.loads(subprocess.check_output(['node','-e',f'console.log(JSON.stringify(require("./world-generator.js").generate({int(seed)})))'],text=True))
print(json.dumps(generate(),indent=2))


In [ ]:
# Inspect a world in a readable form
world = generate(42)
print(world['city'])
factions = {f['id']: f['name'] + ' (' + ' + '.join(f['types']) + ')' for f in world['factions']}
for region in ['Inside City','Outside City','Secret Zone']:
    print('\n' + region)
    for z in world['zones']:
        if z['region'] == region:
            print(z['id'], z['label'], '|', factions.get(z['factionId']), '|', ', '.join(z['functions']), '|', z['reason'], '| host:', z['parentId'])


In [ ]:
# Batch test in one Node process
N = 1000
script = '''const G=require('./world-generator.js');let reg={},types={},bad=0,total=0;for(let i=0;i<%d;i++){const w=G.generate(i);for(const z of w.zones){reg[z.region]=(reg[z.region]||0)+1;types[z.type]=(types[z.type]||0)+1;total++}if(['Authority','Commerce','Residence'].some(k=>w.zoneCoverage['Inside City'][k]<1))bad++}console.log(JSON.stringify({universes:%d,total,reg,types,bad}))''' % (N,N)
print(json.dumps(json.loads(subprocess.check_output(['node','-e',script],text=True)),indent=2))
